In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [4]:
PROJECT_ROOT = Path().resolve().parents[0]
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "telco_processed.csv"
TARGET = "Encoded Churn Label"

In [5]:
df = pd.read_csv(DATA_PATH)

required_columns = {
    TARGET,
    "Churn Label",
    "Churn Value",
    "Tenure Months",
    "Phone Service",
    "Internet Service",
    "Monthly Charges",
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Processed data is missing columns: {sorted(missing_columns)}")

if df[TARGET].isna().any() or not df[TARGET].isin([0, 1]).all():
    raise ValueError(f"{TARGET} must contain non-null binary values.")

class_counts = df[TARGET].value_counts().sort_index()
display(class_counts.rename(index={0: "Retained", 1: "Churned"}))

if set(class_counts.index) != {0, 1}:
    raise ValueError(
        "Both churn classes are required. Correct and rerun notebook 03 before "
        "feature engineering."
    )

if not df["Churn Value"].eq(df[TARGET]).all():
    raise ValueError("Churn Value and Encoded Churn Label do not match.")

print(f"Loaded {len(df):,} rows and {df.shape[1]} columns.")

Encoded Churn Label
Retained    5163
Churned     1869
Name: count, dtype: int64

Loaded 7,032 rows and 22 columns.


# Add customer tenure, household, and service features

In [6]:
df = df.copy()

addon_columns = [
    "Multiple Lines",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies",
]

missing_addons = set(addon_columns).difference(df.columns)
if missing_addons:
    raise ValueError(f"Processed data is missing service columns: {sorted(missing_addons)}")

df["Tenure Group"] = pd.cut(
    df["Tenure Months"],
    bins=[-1, 12, 24, 48, 60, np.inf],
    labels=["0-12", "13-24", "25-48", "49-60", "61+"],
)

df["Has Household"] = (
    df["Partner"].eq("Yes") | df["Dependents"].eq("Yes")
).astype("int8")

df["Service Count"] = (
    df["Phone Service"].eq("Yes").astype("int8")
    + df["Internet Service"].ne("No").astype("int8")
    + df[addon_columns].eq("Yes").sum(axis=1).astype("int8")
)

df["Monthly Charges per Service"] = (
    df["Monthly Charges"] / df["Service Count"].clip(lower=1)
)

display(df[[
    "Tenure Months",
    "Tenure Group",
    "Has Household",
    "Service Count",
    "Monthly Charges",
    "Monthly Charges per Service",
]].head())

,Tenure Months,Tenure Group,Has Household,Service Count,Monthly Charges,Monthly Charges per Service
0,2,0-12,0,4,53.85,13.462500
1,2,0-12,1,2,70.70,35.350000
2,8,0-12,1,6,99.65,16.608333
3,28,25-48,1,7,104.80,14.971429
4,49,49-60,1,7,103.70,14.814286


# Separate predictors and target, then split the data

In [7]:
columns_to_exclude = [TARGET, "Churn Label", "Churn Value"]
X = df.drop(columns=columns_to_exclude)
y = df[TARGET].astype("int8")

numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = [
    column for column in X.columns
    if column not in numeric_features
]

if not numeric_features or not categorical_features:
    raise ValueError("Expected both numeric and categorical predictor columns.")

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y,
)

print(f"Training rows: {len(X_train_raw):,}; test rows: {len(X_test_raw):,}")

Training rows: 5,625; test rows: 1,407


# Scale numeric predictors and one-hot encode categorical predictors
Fitting the transformations on the training set only, then apply them to the test set.

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features,
        ),
    ],
)

X_train_fe = pd.DataFrame(
    preprocessor.fit_transform(X_train_raw),
    index=X_train_raw.index,
    columns=preprocessor.get_feature_names_out(),
)

X_test_fe = pd.DataFrame(
    preprocessor.transform(X_test_raw),
    index=X_test_raw.index,
    columns=preprocessor.get_feature_names_out(),
)

assert X_train_fe.columns.equals(X_test_fe.columns)
assert not X_train_fe.isna().any().any()
assert not X_test_fe.isna().any().any()

print(f"Encoded training features: {X_train_fe.shape}")
print(f"Encoded test features: {X_test_fe.shape}")
display(X_train_fe.head())

Encoded training features: (5625, 54)
Encoded test features: (1407, 54)


,num__Tenure Months,num__Monthly Charges,num__Total Charges,num__Has Household,num__Service Count,num__Monthly Charges per Service,cat__Gender_Female,cat__Gender_Male,cat__Senior Citizen_No,cat__Senior Citizen_Yes,...,cat__Paperless Billing_Yes,cat__Payment Method_Bank transfer (automatic),cat__Payment Method_Credit card (automatic),cat__Payment Method_Electronic check,cat__Payment Method_Mailed check,cat__Tenure Group_0-12,cat__Tenure Group_13-24,cat__Tenure Group_25-48,cat__Tenure Group_49-60,cat__Tenure Group_61+
1695,0.756833,0.424463,0.668412,0.946158,-0.058977,0.376444,1.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
5984,-1.072711,0.515972,-0.786967,-1.056906,1.240285,-1.081065,0.0,1.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
4179,0.350268,1.475974,0.952510,0.946158,1.673372,-0.680662,1.0,0.0,1.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3328,-0.910084,1.015106,-0.555021,-1.056906,0.807197,-0.268414,0.0,1.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
6689,-1.154024,-1.184415,-0.955322,-1.056906,-0.925151,-0.507932,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0


# Save the model-ready splits

In [9]:
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

train_ready = X_train_fe.assign(**{TARGET: y_train})
test_ready = X_test_fe.assign(**{TARGET: y_test})

train_ready.to_csv(INTERIM_DIR / "telco_train_features.csv", index=False)
test_ready.to_csv(INTERIM_DIR / "telco_test_features.csv", index=False)

print(f"Saved feature splits to {INTERIM_DIR}")

Saved feature splits to C:\Users\Deputy\Github\customer-churn-prediction\data\interim
